# Notebook 04 — Annotation: IDG/Pharos, GTEx Aging, Mechanism Clustering

**Goal:** Enrich the ranked compound list with three annotation layers:

1. **IDG/Pharos target annotation** — for each compound, retrieve known targets
   and Target Development Level (TDL: Tclin/Tchem/Tbio/Tdark). Flag compounds
   hitting understudied targets (Tbio/Tdark) — these represent novel exercise-
   mimetic biology rather than rediscovery of known biology.

2. **GTEx aging concordance** — does the compound's signature oppose the tissue-
   matched aging signature? Compounds that both mimic exercise AND oppose aging
   get a composite boost. Anti-aging concordance is the most biologically
   meaningful filter in this pipeline.

3. **Mechanism-of-action clustering** — group top compounds by MOA so the
   final ranking presents mechanisms, not just individual compounds. Five PPARδ
   agonists in the top 20 should collapse to one entry.

**Outputs:**
- `data/processed/idg_annotations.parquet`
- `data/processed/aging_concordance.parquet`
- `data/processed/mechanism_clusters.parquet`
- `data/processed/annotated_compounds.parquet`

In [ ]:
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
from src import signatures as sig
from src import annotation as ann
from src import connectivity as conn

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# ── Configuration ─────────────────────────────────────────────────────────────
TOP_N_ANNOTATE = 500      # Annotate top N from meta-ranking
PHAROS_BATCH   = 50       # Compounds per Pharos query session

TISSUES      = list(sig.TISSUES.keys())
TIMEPOINT    = '8w'

PROCESSED_DIR = Path('../data/processed')
EXTERNAL_DIR  = Path('../data/external')
FIGURES_DIR   = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Top N to annotate: {TOP_N_ANNOTATE}')

In [ ]:
# ── Load ranked compound list from notebook 02 ────────────────────────────────

meta_path = PROCESSED_DIR / 'ranked_compounds_meta.parquet'
if not meta_path.exists():
    raise FileNotFoundError('ranked_compounds_meta.parquet not found — run notebook 02 first.')

ranked_meta = pd.read_parquet(meta_path)
top_compounds = ranked_meta.head(TOP_N_ANNOTATE)['pert_iname'].tolist()

print(f'Loaded {len(ranked_meta)} meta-ranked compounds')
print(f'Annotating top {len(top_compounds)} compounds')
print(f'\nTop 10:')
print(ranked_meta.head(10)[['pert_iname', 'mean_score', 'n_tissues', 'moa']].to_string(index=False))

In [ ]:
# ── Layer 1: IDG / Pharos target annotation ───────────────────────────────────

idg_cache = PROCESSED_DIR / 'idg_annotations.parquet'

print(f'Querying Pharos for {len(top_compounds)} compounds …')
print('(Cached after first run)')

idg_raw = ann.annotate_idg(
    top_compounds,
    cache_path=idg_cache,
)

idg_summary = ann.summarize_idg_per_compound(idg_raw)

print(f'\nIDG annotation complete: {len(idg_summary)} compounds annotated')
print(f'Compounds with novel targets (Tbio/Tdark): '
      f'{idg_summary["has_novel_target"].sum()}')
print()

# TDL distribution
tdl_counts = idg_raw['tdl'].value_counts().reindex(
    ['Tclin', 'Tchem', 'Tbio', 'Tdark', None], fill_value=0
)
print('TDL distribution of targets:')
print(tdl_counts)

# Flag novel hits
novel = idg_summary[idg_summary['has_novel_target']].merge(
    ranked_meta[['pert_iname', 'meta_rank']],
    left_on='compound', right_on='pert_iname', how='left'
).sort_values('meta_rank')

print(f'\nTop novel-target compounds (Tbio/Tdark):')
if not novel.empty:
    print(novel[['compound', 'meta_rank', 'novel_targets', 'best_tdl_label']].head(10).to_string(index=False))

In [ ]:
# ── IDG annotation visualisation ──────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# TDL pie chart
ax = axes[0]
tdl_labels = ['Tclin', 'Tchem', 'Tbio', 'Tdark']
tdl_vals   = [idg_raw['tdl'].eq(t).sum() for t in tdl_labels]
colors     = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
wedges, texts, autotexts = ax.pie(
    tdl_vals, labels=tdl_labels, colors=colors,
    autopct='%1.1f%%', startangle=140
)
ax.set_title('Target Development Level (TDL)\nof top-ranked compound targets')

# Novel target enrichment by meta-rank
ax = axes[1]
idg_merged = ranked_meta.head(TOP_N_ANNOTATE).merge(
    idg_summary[['compound', 'has_novel_target']],
    left_on='pert_iname', right_on='compound', how='left'
)
idg_merged['has_novel_target'] = idg_merged['has_novel_target'].fillna(False)
idg_merged['rank_bin'] = pd.cut(idg_merged['meta_rank'], bins=10)

novel_by_rank = idg_merged.groupby('rank_bin')['has_novel_target'].mean() * 100
novel_by_rank.plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
ax.set_title('% compounds with novel (Tbio/Tdark) targets\nby meta-rank bin')
ax.set_xlabel('Meta-rank bin')
ax.set_ylabel('% with novel target')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_idg_annotation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved IDG annotation figure.')

In [ ]:
# ── Layer 2: GTEx aging concordance ──────────────────────────────────────────
# Load tissue-matched aging gene sets and compute anti-aging concordance.

aging_sigs = {}
for tissue in TISSUES:
    aging_gs = ann.load_aging_signatures(tissue, EXTERNAL_DIR)
    aging_sigs[tissue] = aging_gs
    print(f'{tissue}: {len(aging_gs["up"])} aging-up, {len(aging_gs["down"])} aging-down genes')

In [ ]:
# Compute anti-aging concordance per compound per tissue.
# Uses enrichment scores as proxy (full gene-level mode requires GCTX).

aging_records = []

for tissue in TISSUES:
    raw_path = PROCESSED_DIR / f'lincs_raw_{tissue}.parquet'
    if not raw_path.exists():
        print(f'Skipping {tissue}: raw LINCS data not available.')
        continue

    lincs_raw = pd.read_parquet(raw_path)
    ranked_tissue_path = PROCESSED_DIR / f'lincs_ranked_{tissue}.parquet'
    if not ranked_tissue_path.exists():
        continue
    ranked_tissue = pd.read_parquet(ranked_tissue_path)

    aging_up = aging_sigs[tissue]['up']
    aging_dn = aging_sigs[tissue]['down']

    for compound in top_compounds[:200]:  # top-200 for speed
        compound_sigs = lincs_raw[
            lincs_raw['pert_iname'].str.lower() == compound.lower()
        ]
        if compound_sigs.empty:
            continue

        # Build a proxy gene-level signature from overlap genes
        # Positive overlap_up genes → should be UP in compound (anti-aging = compound DOWN)
        all_genes_up   = set()
        all_genes_down = set()
        for _, row in compound_sigs.iterrows():
            for direction, gene_set in [('up', all_genes_up), ('down', all_genes_down)]:
                raw_g = row.get(f'overlap_{direction}', '')
                if isinstance(raw_g, str) and raw_g:
                    gene_set.update(raw_g.split('|'))

        # Anti-aging score: fraction of aging-up genes in compound-down,
        # plus fraction of aging-dn genes in compound-up
        aging_up_set = set(aging_up)
        aging_dn_set = set(aging_dn)

        n_anti_up = len(all_genes_down & aging_up_set)
        n_anti_dn = len(all_genes_up   & aging_dn_set)
        denom = max(len(aging_up_set) + len(aging_dn_set), 1)

        # Also use the enrichment score itself as a simple proxy
        score_col = 'median_score' if 'median_score' in ranked_tissue.columns else ranked_tissue.columns[1]
        comp_row = ranked_tissue[ranked_tissue['pert_iname'].str.lower() == compound.lower()]
        enrich_score = comp_row[score_col].iloc[0] if not comp_row.empty else 0.0

        anti_aging_score = (n_anti_up + n_anti_dn) / denom

        aging_records.append({
            'compound': compound,
            'tissue': tissue,
            'anti_aging_score': anti_aging_score,
            'n_anti_up': n_anti_up,
            'n_anti_dn': n_anti_dn,
            'enrich_score': enrich_score,
        })

aging_df = pd.DataFrame(aging_records)
if not aging_df.empty:
    aging_df.to_parquet(PROCESSED_DIR / 'aging_concordance.parquet', index=False)
    print(f'Aging concordance computed for {aging_df["compound"].nunique()} compounds')
    print(aging_df.groupby('tissue')['anti_aging_score'].describe())
else:
    print('No aging concordance data computed.')

In [ ]:
# ── Aging concordance figure ──────────────────────────────────────────────────

if not aging_df.empty:
    # Average anti-aging score across tissues per compound
    aging_agg = (
        aging_df.groupby('compound')['anti_aging_score']
        .mean()
        .reset_index()
        .rename(columns={'anti_aging_score': 'mean_anti_aging'})
        .sort_values('mean_anti_aging', ascending=False)
    )

    # Merge with enrichment rank
    aging_agg = aging_agg.merge(
        ranked_meta[['pert_iname', 'meta_rank', 'mean_score']],
        left_on='compound', right_on='pert_iname', how='left'
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Anti-aging score vs enrichment score scatter
    ax = axes[0]
    ax.scatter(
        aging_agg['mean_score'], aging_agg['mean_anti_aging'],
        alpha=0.5, s=20, color='teal'
    )
    top_hits = aging_agg.query(
        'mean_anti_aging > mean_anti_aging.quantile(0.85) and '
        'mean_score > mean_score.quantile(0.85)'
    )
    for _, row in top_hits.head(10).iterrows():
        ax.annotate(row['compound'], (row['mean_score'], row['mean_anti_aging']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points')
    ax.set_xlabel('Enrichment score (exercise mimicry)')
    ax.set_ylabel('Anti-aging concordance score')
    ax.set_title('Exercise mimicry vs Anti-aging concordance')

    # Top 20 anti-aging compounds
    ax = axes[1]
    top20_aging = aging_agg.head(20)
    bars = ax.barh(top20_aging['compound'][::-1],
                   top20_aging['mean_anti_aging'][::-1],
                   color='teal', alpha=0.8)
    ax.set_xlabel('Anti-aging concordance score')
    ax.set_title('Top-20 compounds by anti-aging concordance')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '04_aging_concordance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved aging concordance figure.')

In [ ]:
# ── Layer 3: Mechanism-of-action clustering ───────────────────────────────────

moa_df = ann.cluster_by_mechanism(ranked_meta, moa_col='moa', target_col='target', top_n=200)

if not moa_df.empty:
    moa_df.to_parquet(PROCESSED_DIR / 'mechanism_clusters.parquet', index=False)
    print(f'MOA clusters: {len(moa_df)} distinct mechanisms in top-200 compounds')
    print()
    print(moa_df.head(15).to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, max(4, len(moa_df.head(15)) * 0.4)))
    top_moa = moa_df.head(15)
    ax.barh(top_moa['mechanism_cluster'][::-1], top_moa['n_compounds'][::-1],
            color='#5c85d6', edgecolor='white')
    ax.set_xlabel('Number of top-200 compounds')
    ax.set_title('Mechanism-of-action clusters in top-200 exercise mimetics')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '04_moa_clusters.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved MOA clustering figure.')
else:
    print('MOA data not available — check that LINCS metadata contains moa/target columns.')

In [ ]:
# ── Merge all annotation layers ────────────────────────────────────────────────

annotated = ranked_meta.copy()

# Add IDG
if not idg_summary.empty:
    annotated = annotated.merge(
        idg_summary[['compound', 'best_tdl_label', 'has_novel_target', 'novel_targets', 'n_targets']],
        left_on='pert_iname', right_on='compound', how='left'
    ).drop(columns=['compound'], errors='ignore')

# Add aging concordance (averaged across tissues)
if not aging_df.empty:
    aging_per_compound = (
        aging_df.groupby('compound')['anti_aging_score']
        .mean().reset_index()
        .rename(columns={'anti_aging_score': 'anti_aging_score'})
    )
    annotated = annotated.merge(
        aging_per_compound, left_on='pert_iname', right_on='compound', how='left'
    ).drop(columns=['compound'], errors='ignore')

annotated.to_parquet(PROCESSED_DIR / 'annotated_compounds.parquet', index=False)
print(f'Saved annotated_compounds.parquet ({len(annotated)} rows, {len(annotated.columns)} columns)')
print(f'Columns: {list(annotated.columns)}')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

print('=== Notebook 04 complete ===')
print()
print(f'IDG annotations: {len(idg_summary)} compounds')
print(f'  Novel-target compounds (Tbio/Tdark): {idg_summary["has_novel_target"].sum() if not idg_summary.empty else 0}')
print(f'Aging concordance: {aging_df["compound"].nunique() if not aging_df.empty else 0} compounds')
print(f'MOA clusters: {len(moa_df) if not moa_df.empty else 0} mechanisms')
print()
print('→ Next: run notebook 05_final_ranking.ipynb')